In [ ]:
pip install langchain langchain-openai langchain-community duckduckgo-search langgraph langchain-mcp-adapters

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

# 1. Define the AgentState
class AgentState(TypedDict):
    question: str
    answer: str
    feedback: str
    is_acceptable: bool
    iterations: int

llm = ChatOpenAI(temperature=0.7, model="gpt-4o-mini")

# 2. Node: Generate initial answer
def generate_node(state: AgentState):
    prompt = f"Answer the following research question briefly: {state['question']}"
    response = llm.invoke(prompt)
    return {"answer": response.content, "iterations": state.get("iterations", 0) + 1}

# 3. Node: Evaluate the answer
def evaluate_node(state: AgentState):
    answer = state.get("answer", "")
    word_count = len(answer.split())

    # Heuristic check: Answer must be detailed (> 30 words) and contain the word "research"
    if word_count > 30 and "research" in answer.lower():
        return {"is_acceptable": True, "feedback": "Pass"}
    else:
        return {
            "is_acceptable": False,
            "feedback": f"The answer is too short ({word_count} words) or missing the keyword 'research'. Please expand."
        }

# 4. Node: Refine the answer based on feedback
def refine_node(state: AgentState):
    prompt = f"""
    Question: {state['question']}
    Previous Answer: {state['answer']}
    Feedback: {state['feedback']}

    Please improve the answer to resolve the feedback.
    """
    response = llm.invoke(prompt)
    return {"answer": response.content, "iterations": state["iterations"] + 1}

# 5. Conditional Edge Logic
def route_evaluation(state: AgentState):
    if state["is_acceptable"]:
        return END
    else:
        return "refine"

def run_cyclic_graph():
    # Build the graph
    workflow = StateGraph(AgentState)
    workflow.add_node("generate", generate_node)
    workflow.add_node("evaluate", evaluate_node)
    workflow.add_node("refine", refine_node)

    workflow.set_entry_point("generate")
    workflow.add_edge("generate", "evaluate")
    workflow.add_conditional_edges("evaluate", route_evaluation, {END: END, "refine": "refine"})
    workflow.add_edge("refine", "evaluate") # Close the loop

    app = workflow.compile()

    # Run the graph
    initial_input = {"question": "Explain the significance of quantum computing.", "iterations": 0}
    final_state = app.invoke(initial_input)

    print("\n--- Final Agent State ---")
    for key, value in final_state.items():
        print(f"{key.capitalize()}: {value}\n")

if __name__ == "__main__":
    run_cyclic_graph()

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.